# Multi-task нейросеть intent + topic для DialogSum-RU

Этот блокнот реализует **многозадачную (multi-task) нейросетевую модель**, которая
одновременно решает две задачи на репликах диалогов DialogSum-RU:

1. **Основная задача — intent detection.** Определение намерения говорящего по реплике
   (`intent_label`).
2. **Вспомогательная задача — topic cluster prediction.** Предсказание идентификатора
   тематического кластера `cluster_id`, полученного эмбеддинговой кластеризацией в
   блокноте `08_topic_modeling_dialogsum_ru.ipynb`.

Идея в том, что вспомогательная задача предсказания тематического кластера выступает
**индуктивным смещением (inductive bias)** для энкодера: модель вынуждена выучивать
представления, чувствительные не только к коммуникативному намерению, но и к смысловой
теме реплики. Это особенно полезно в условиях **слабой разметки intent-меток** (weak
labels), полученных эвристиками в блокнотах `09`/`10`.

### План блокнота

1. Импорты и конфигурация.
2. Монтирование Google Drive и пути проекта.
3. Загрузка и подготовка данных (utterance-level intents + cluster_id).
4. Dataset и DataLoader'ы под HuggingFace токенизатор.
5. Архитектура multi-task модели и single-task baseline.
6. Утилиты обучения и валидации.
7. Обучение single-task baseline (только intent).
8. Обучение multi-task модели (intent + topic).
9. Сравнение метрик single-task vs multi-task.
10. Анализ ошибок и визуализации.
11. Сохранение артефактов в Google Drive.
12. Выводы и обсуждение.

### Замечания

- Используются **PyTorch + HuggingFace Transformers** (без Keras/TensorFlow).
- Платные внешние API не задействованы.
- Код проверяет наличие GPU (`torch.cuda`) и корректно откатывается на CPU.
- В реальной работе при отсутствии GPU стоит уменьшить `EPOCHS` и/или задать
  `MAX_SAMPLES` для отладки.


## Ячейка 1 — Импорты и глобальные настройки

Раскомментируйте установку пакетов, если они отсутствуют в текущем окружении (Colab / Perplexity Compute).

In [ ]:
# cell 1: imports and config
# Если пакеты не установлены в Colab/Perplexity Compute, раскомментируйте строку ниже:
# !pip install -q transformers accelerate scikit-learn pandas numpy matplotlib seaborn tqdm joblib pyarrow

import os
import re
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

# ----------------------------- Конфигурация -----------------------------
RANDOM_STATE = 42

# Базовая модель-энкодер. Можно заменить на:
#   'DeepPavlov/rubert-base-cased' или 'xlm-roberta-base'
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

# Веса задач в совместном лоссе
LAMBDA_INTENT = 1.0
LAMBDA_TOPIC = 0.5

# Подготовка данных
TOP_N_CLUSTERS = 10        # Кол-во наиболее частых не-шумовых кластеров для topic head
MIN_INTENT_COUNT = 20      # Минимальная встречаемость intent-класса (иначе фильтруем)
MAX_SAMPLES = None         # Ограничение размера выборки для отладки (None = без ограничений)

# Доп. опции
FREEZE_ENCODER = False     # Если True, обучается только head; полезно на CPU

# ----------------------------- Воспроизводимость -----------------------------
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch версия: {torch.__version__}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
print(f'Используемое устройство: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Предупреждение: тренировка на CPU будет медленной. Рассмотрите уменьшение EPOCHS или установку MAX_SAMPLES.')


## Ячейка 2 — Google Drive и пути проекта

Монтируем Google Drive (в Colab) и фиксируем пути к таблицам, фигурам и моделям проекта.

In [ ]:
# cell 2: Google Drive and project paths
try:
    from google.colab import drive  # type: ignore
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print('Google Drive уже смонтирован.')
except ModuleNotFoundError:
    print('google.colab не найден: пропускаем монтирование Google Drive.')

BASE_DIR = Path('/content/drive/MyDrive/russian-dialogue-intent-thesis')
TABLES_DIR = BASE_DIR / 'results' / 'tables'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'
MODELS_DIR = BASE_DIR / 'models' / 'multitask_intent_topic'

if not BASE_DIR.exists():
    print(f'Предупреждение: базовая директория проекта {BASE_DIR} не существует.')
    print('Если вы работаете локально или без Google Drive, обновите пути выше вручную.')
else:
    print(f'BASE_DIR найдена: {BASE_DIR}')

# Создаём только подкаталоги для результатов и моделей (не весь BASE_DIR молча)
for d in [FIGURES_DIR, MODELS_DIR]:
    try:
        d.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        print(f'Не удалось создать {d}: {e}')

print(f'TABLES_DIR:  {TABLES_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')
print(f'MODELS_DIR:  {MODELS_DIR}')


## Ячейка 3 — Загрузка и подготовка данных

Логика загрузки:

1. **Предпочтительный путь:** weak utterance dataset из блокнотов `09`/`10`
   (`dialogsum_ru_intent_utterances_weak.parquet`/`.csv`).
2. **Fallback:** восстановление из topic clusters файла блокнота `08`
   (`dialogsum_ru_topic_clusters_sota.parquet`/`.csv`) с разбиением `dialogue` на
   реплики `#Person1#`/`#Person2#` и присвоением rule-based intent-меток.
3. Если нет ни того, ни другого — выдаём осмысленную ошибку с подсказкой запустить `08`/`09`/`10`.


In [ ]:
# cell 3: load and prepare data
WEAK_PARQUET = TABLES_DIR / 'dialogsum_ru_intent_utterances_weak.parquet'
WEAK_CSV = TABLES_DIR / 'dialogsum_ru_intent_utterances_weak.csv'
TOPIC_PARQUET = TABLES_DIR / 'dialogsum_ru_topic_clusters_sota.parquet'
TOPIC_CSV = TABLES_DIR / 'dialogsum_ru_topic_clusters_sota.csv'

USED_FALLBACK = False


def _read_any(parquet_path: Path, csv_path: Path):
    if parquet_path.exists():
        print(f'Читаю parquet: {parquet_path}')
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        print(f'Читаю csv: {csv_path}')
        return pd.read_csv(csv_path)
    return None


# ----------------------------- Rule-based intent (fallback) -----------------------------
_INTENT_RULES = [
    ('greeting', re.compile(r'\b(привет|здравствуй|здравствуйте|добрый\s+(день|вечер|утро)|hello|hi)\b', re.I)),
    ('farewell', re.compile(r'\b(пока|до\s+свидан|удач|всего\s+доброго|bye|goodbye)\b', re.I)),
    ('thanks', re.compile(r'\b(спасибо|благодар|thanks|thank you)\b', re.I)),
    ('confirmation', re.compile(r'^(\s*)(да|конечно|разумеется|хорошо|ладно|ок|окей|yes|sure|ok|okay)\b', re.I)),
    ('rejection', re.compile(r'^(\s*)(нет|не\s+могу|не\s+хочу|no|nope)\b', re.I)),
    ('clarification_request', re.compile(r'(что\s+вы\s+имеете|что\s+значит|повторите|уточните|можете\s+пояснить|what\s+do\s+you\s+mean)', re.I)),
    ('service_request', re.compile(r'(закажу|оформ|забронируй|подключите|сделайте|выпиш|please\s+(do|set|book))', re.I)),
    ('purchase_or_booking_request', re.compile(r'(купить|приобрест|заказ|бронир|оплат|купи|book|purchase|order)', re.I)),
    ('complaint', re.compile(r'(жалоб|недоволен|возмут|ужасн|отвратитель|complaint|terrible|awful)', re.I)),
    ('problem_report', re.compile(r'(не\s+работает|сломал|ошибк|проблем|broken|not\s+working|error)', re.I)),
    ('arrangement', re.compile(r'(встрет|договор|назнач|перенес|встреча|let\'s\s+meet|schedule)', re.I)),
    ('suggestion_or_recommendation', re.compile(r'(рекоменду|советую|предлага|стоит\s+попробовать|recommend|suggest)', re.I)),
    ('opinion_or_preference', re.compile(r'(нравится|предпочит|думаю,?\s+что|по\s+моему\s+мнению|i\s+think|prefer|like)', re.I)),
    ('informational_request', re.compile(r'^[^?]*\?\s*$|\b(кто|что|когда|где|как|почему|сколько|what|how|when|where|why)\b.*\?', re.I)),
]

INTENT_CLASSES = [
    'greeting', 'thanks', 'farewell', 'informational_request', 'clarification_request',
    'service_request', 'purchase_or_booking_request', 'complaint', 'problem_report',
    'arrangement', 'confirmation', 'rejection', 'suggestion_or_recommendation',
    'opinion_or_preference', 'other',
]


def rule_based_intent(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return 'other'
    for label, pattern in _INTENT_RULES:
        if pattern.search(text):
            return label
    return 'other'


_SPEAKER_RE = re.compile(r'#Person([12])#\s*[:：]?\s*', re.IGNORECASE)


def split_dialogue_to_utterances(dialogue: str):
    if not isinstance(dialogue, str):
        return []
    parts = _SPEAKER_RE.split(dialogue)
    utterances = []
    i = 1
    while i + 1 < len(parts):
        speaker = f'Person{parts[i]}'
        text = parts[i + 1].strip()
        if text:
            utterances.append((speaker, text))
        i += 2
    return utterances


def build_fallback_from_topics(topics_df: pd.DataFrame) -> pd.DataFrame:
    print('Восстанавливаю weak utterance dataset из topic clusters (fallback).')
    dialogue_col = 'dialogue' if 'dialogue' in topics_df.columns else (
        'dialogue_ru' if 'dialogue_ru' in topics_df.columns else None
    )
    if dialogue_col is None:
        raise ValueError('В topic clusters не найдена колонка dialogue/dialogue_ru — невозможно построить fallback.')
    if 'cluster_id' not in topics_df.columns:
        raise ValueError('В topic clusters отсутствует cluster_id — fallback невозможен.')

    id_col = 'dialogue_id' if 'dialogue_id' in topics_df.columns else None
    name_col = 'cluster_name' if 'cluster_name' in topics_df.columns else None

    rows = []
    for _, r in topics_df.iterrows():
        utts = split_dialogue_to_utterances(r[dialogue_col])
        for idx, (sp, txt) in enumerate(utts):
            rows.append({
                'dialogue_id': r[id_col] if id_col else None,
                'utterance_idx': idx,
                'speaker': sp,
                'utterance_text': txt,
                'cluster_id': r['cluster_id'],
                'cluster_name': r[name_col] if name_col else None,
            })
    df = pd.DataFrame(rows)
    df['intent_label'] = df['utterance_text'].apply(rule_based_intent)
    df['is_fallback_weak'] = True
    return df


# ----------------------------- Основная загрузка -----------------------------
df = _read_any(WEAK_PARQUET, WEAK_CSV)
if df is None:
    topics_df = _read_any(TOPIC_PARQUET, TOPIC_CSV)
    if topics_df is None:
        raise FileNotFoundError(
            'Не найдены ни weak utterance dataset, ни topic clusters. '
            'Сначала запустите блокноты 08_topic_modeling_dialogsum_ru.ipynb, '
            '09_intent_modeling_dialogsum_ru.ipynb или 10_intent_manual_validation_dialogsum_ru.ipynb.'
        )
    df = build_fallback_from_topics(topics_df)
    USED_FALLBACK = True
else:
    df['is_fallback_weak'] = False

required_cols = ['utterance_text', 'intent_label', 'cluster_id']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'В исходных данных отсутствуют обязательные колонки: {missing}')

print(f'Загружено строк: {len(df)} | fallback weak: {USED_FALLBACK}')
print('Колонки:', list(df.columns))

# ----------------------------- Очистка -----------------------------
df = df.dropna(subset=['utterance_text', 'intent_label', 'cluster_id']).copy()
df['utterance_text'] = df['utterance_text'].astype(str).str.strip()
df = df[df['utterance_text'].str.len() > 0]
df['cluster_id'] = df['cluster_id'].astype(int)

# Для topic head убираем шумовой кластер (-1) и оставляем top-N не-шумовых
non_noise = df[df['cluster_id'] != -1]
top_clusters = non_noise['cluster_id'].value_counts().head(TOP_N_CLUSTERS).index.tolist()
print(f'Топ-{TOP_N_CLUSTERS} не-шумовых кластеров (cluster_id): {top_clusters}')
df = df[df['cluster_id'].isin(top_clusters)].copy()

# Фильтрация редких intent-классов
intent_counts = df['intent_label'].value_counts()
rare = intent_counts[intent_counts < MIN_INTENT_COUNT].index.tolist()
if rare:
    print(f'Отфильтровываю редкие intent-классы (<{MIN_INTENT_COUNT}): {rare}')
    df = df[~df['intent_label'].isin(rare)].copy()

if MAX_SAMPLES is not None and len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Применён MAX_SAMPLES={MAX_SAMPLES}, итоговый размер: {len(df)}')

# ----------------------------- Кодирование меток -----------------------------
intent_encoder = LabelEncoder()
df['intent_id'] = intent_encoder.fit_transform(df['intent_label'])

topic_encoder = LabelEncoder()
df['topic_id'] = topic_encoder.fit_transform(df['cluster_id'])

NUM_INTENTS = len(intent_encoder.classes_)
NUM_TOPICS = len(topic_encoder.classes_)
print(f'Кол-во intent-классов: {NUM_INTENTS} -> {list(intent_encoder.classes_)}')
print(f'Кол-во topic-классов: {NUM_TOPICS} -> {list(topic_encoder.classes_)}')

# ----------------------------- Train/val/test split -----------------------------
try:
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['intent_id'], random_state=RANDOM_STATE,
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['intent_id'], random_state=RANDOM_STATE,
    )
except ValueError as e:
    print(f'Стратификация не удалась ({e}), повторяю без стратификации.')
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=RANDOM_STATE)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Размеры: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
print('Распределение intent на train:')
print(train_df['intent_label'].value_counts())


## Ячейка 4 — Dataset и DataLoader

Оборачиваем данные в `torch.utils.data.Dataset` и подключаем HuggingFace токенизатор.

In [ ]:
# cell 4: dataset and dataloaders
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Токенизатор {MODEL_NAME} загружен. Vocab size: {tokenizer.vocab_size}')


class DialogSumIntentTopicDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts = dataframe['utterance_text'].tolist()
        self.intents = dataframe['intent_id'].tolist()
        self.topics = dataframe['topic_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'intent_label': torch.tensor(self.intents[idx], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[idx], dtype=torch.long),
        }


train_ds = DialogSumIntentTopicDataset(train_df, tokenizer)
val_ds = DialogSumIntentTopicDataset(val_df, tokenizer)
test_ds = DialogSumIntentTopicDataset(test_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Батчей: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')


## Ячейка 5 — Архитектура моделей

- **MultiTaskIntentTopicModel** — общий энкодер (HuggingFace `AutoModel`), mean pooling
  по attention mask, общая проекция и две головы: `intent_head` и `topic_head`.
- **SingleTaskIntentModel** — тот же энкодер + проекция + только `intent_head`. Служит
  baseline для сравнения.

Совместный лосс: `L = lambda_intent * CE(intent) + lambda_topic * CE(topic)`.


In [ ]:
# cell 5: multitask model definition
from transformers import AutoModel


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class MultiTaskIntentTopicModel(nn.Module):
    def __init__(self, base_model_name: str, num_intents: int, num_topics: int,
                 dropout: float = 0.1, freeze_encoder: bool = False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden = self.encoder.config.hidden_size

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        self.shared_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
        )
        self.intent_head = nn.Linear(hidden, num_intents)
        self.topic_head = nn.Linear(hidden, num_topics)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, intent_labels=None, topic_labels=None,
                lambda_intent: float = LAMBDA_INTENT, lambda_topic: float = LAMBDA_TOPIC):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(out.last_hidden_state, attention_mask)
        shared = self.shared_proj(pooled)
        intent_logits = self.intent_head(shared)
        topic_logits = self.topic_head(shared)

        result = {'intent_logits': intent_logits, 'topic_logits': topic_logits}
        if intent_labels is not None and topic_labels is not None:
            l_intent = self.loss_fn(intent_logits, intent_labels)
            l_topic = self.loss_fn(topic_logits, topic_labels)
            result['loss_intent'] = l_intent
            result['loss_topic'] = l_topic
            result['loss'] = lambda_intent * l_intent + lambda_topic * l_topic
        return result


class SingleTaskIntentModel(nn.Module):
    def __init__(self, base_model_name: str, num_intents: int,
                 dropout: float = 0.1, freeze_encoder: bool = False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden = self.encoder.config.hidden_size

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        self.proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
        )
        self.intent_head = nn.Linear(hidden, num_intents)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, intent_labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(out.last_hidden_state, attention_mask)
        proj = self.proj(pooled)
        intent_logits = self.intent_head(proj)
        result = {'intent_logits': intent_logits}
        if intent_labels is not None:
            result['loss'] = self.loss_fn(intent_logits, intent_labels)
            result['loss_intent'] = result['loss']
        return result


## Ячейка 6 — Утилиты обучения

Функции `train_one_epoch` / `evaluate` и обёртка `fit_model`. Лучшая эпоха выбирается по `val_intent_f1_macro`.

In [ ]:
# cell 6: training utilities
from transformers import get_linear_schedule_with_warmup
import copy


def train_one_epoch(model, loader, optimizer, scheduler, device, multitask: bool):
    model.train()
    total_loss = 0.0
    total_intent_loss = 0.0
    total_topic_loss = 0.0
    n_batches = 0
    pbar = tqdm(loader, desc='train', leave=False)
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        intent_labels = batch['intent_label'].to(device)
        topic_labels = batch['topic_label'].to(device)

        if multitask:
            out = model(input_ids, attention_mask,
                        intent_labels=intent_labels, topic_labels=topic_labels)
        else:
            out = model(input_ids, attention_mask, intent_labels=intent_labels)

        loss = out['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        total_intent_loss += out.get('loss_intent', loss).item()
        if 'loss_topic' in out:
            total_topic_loss += out['loss_topic'].item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    return {
        'train_loss': total_loss / max(n_batches, 1),
        'train_intent_loss': total_intent_loss / max(n_batches, 1),
        'train_topic_loss': total_topic_loss / max(n_batches, 1) if multitask else None,
    }


@torch.no_grad()
def evaluate(model, loader, device, multitask: bool):
    model.eval()
    all_intent_preds, all_intent_labels = [], []
    all_topic_preds, all_topic_labels = [], []
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        intent_labels = batch['intent_label'].to(device)
        topic_labels = batch['topic_label'].to(device)

        if multitask:
            out = model(input_ids, attention_mask,
                        intent_labels=intent_labels, topic_labels=topic_labels)
        else:
            out = model(input_ids, attention_mask, intent_labels=intent_labels)

        if 'loss' in out:
            total_loss += out['loss'].item()
        n_batches += 1

        intent_preds = out['intent_logits'].argmax(dim=-1).cpu().numpy()
        all_intent_preds.extend(intent_preds.tolist())
        all_intent_labels.extend(intent_labels.cpu().numpy().tolist())

        if multitask and 'topic_logits' in out:
            topic_preds = out['topic_logits'].argmax(dim=-1).cpu().numpy()
            all_topic_preds.extend(topic_preds.tolist())
            all_topic_labels.extend(topic_labels.cpu().numpy().tolist())

    metrics = {
        'val_loss': total_loss / max(n_batches, 1),
        'intent_accuracy': accuracy_score(all_intent_labels, all_intent_preds),
        'intent_f1_macro': f1_score(all_intent_labels, all_intent_preds, average='macro', zero_division=0),
        'intent_f1_micro': f1_score(all_intent_labels, all_intent_preds, average='micro', zero_division=0),
        'intent_f1_weighted': f1_score(all_intent_labels, all_intent_preds, average='weighted', zero_division=0),
    }
    if multitask and all_topic_labels:
        metrics['topic_accuracy'] = accuracy_score(all_topic_labels, all_topic_preds)
        metrics['topic_f1_macro'] = f1_score(all_topic_labels, all_topic_preds, average='macro', zero_division=0)
    metrics['_intent_preds'] = all_intent_preds
    metrics['_intent_labels'] = all_intent_labels
    metrics['_topic_preds'] = all_topic_preds
    metrics['_topic_labels'] = all_topic_labels
    return metrics


def fit_model(model, train_loader, val_loader, device, multitask: bool,
              epochs: int = EPOCHS, lr: float = LR):
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    history = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_stats = train_one_epoch(model, train_loader, optimizer, scheduler, device, multitask)
        val_stats = evaluate(model, val_loader, device, multitask)
        log = {**train_stats,
               **{k: v for k, v in val_stats.items() if not k.startswith('_')},
               'epoch': epoch}
        history.append(log)
        print(
            f"Epoch {epoch}/{epochs} | train_loss={log['train_loss']:.4f} | "
            f"val_loss={log['val_loss']:.4f} | intent_acc={log['intent_accuracy']:.4f} | "
            f"intent_f1_macro={log['intent_f1_macro']:.4f}"
            + (f" | topic_acc={log.get('topic_accuracy', 0):.4f}" if multitask else '')
        )
        if val_stats['intent_f1_macro'] > best_f1:
            best_f1 = val_stats['intent_f1_macro']
            best_state = copy.deepcopy(model.state_dict())
            print(f'  -> новый лучший val_intent_f1_macro={best_f1:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_state


## Ячейка 7 — Обучение single-task baseline

Тренируем baseline-модель, которая решает только intent detection. Это нижняя планка для сравнения.

In [ ]:
# cell 7: train single-task baseline
print('=== Обучение SingleTaskIntentModel (baseline) ===')
single_model = SingleTaskIntentModel(
    base_model_name=MODEL_NAME,
    num_intents=NUM_INTENTS,
    freeze_encoder=FREEZE_ENCODER,
).to(DEVICE)
n_params = sum(p.numel() for p in single_model.parameters() if p.requires_grad)
print(f'Обучаемых параметров: {n_params:,}')

single_model, single_history, single_best_state = fit_model(
    single_model, train_loader, val_loader, DEVICE, multitask=False,
)

print('\n--- Финальная валидация single-task ---')
single_val_metrics = evaluate(single_model, val_loader, DEVICE, multitask=False)
for k, v in single_val_metrics.items():
    if not k.startswith('_') and isinstance(v, float):
        print(f'  {k}: {v:.4f}')


## Ячейка 8 — Обучение multi-task модели

Тренируем `MultiTaskIntentTopicModel` с двумя головами и совместным лоссом.

In [ ]:
# cell 8: train multi-task model
print('=== Обучение MultiTaskIntentTopicModel ===')
multi_model = MultiTaskIntentTopicModel(
    base_model_name=MODEL_NAME,
    num_intents=NUM_INTENTS,
    num_topics=NUM_TOPICS,
    freeze_encoder=FREEZE_ENCODER,
).to(DEVICE)
n_params = sum(p.numel() for p in multi_model.parameters() if p.requires_grad)
print(f'Обучаемых параметров: {n_params:,}')

multi_model, multi_history, multi_best_state = fit_model(
    multi_model, train_loader, val_loader, DEVICE, multitask=True,
)

print('\n--- Финальная валидация multi-task ---')
multi_val_metrics = evaluate(multi_model, val_loader, DEVICE, multitask=True)
for k, v in multi_val_metrics.items():
    if not k.startswith('_') and isinstance(v, float):
        print(f'  {k}: {v:.4f}')


## Ячейка 9 — Оценка на тестовой выборке и сравнение

Считаем метрики обеих моделей на отложенном `test`-сете, печатаем classification report
по intent и сохраняем сводную таблицу сравнения.


In [ ]:
# cell 9: evaluation and comparison
print('=== Тестовая оценка ===')
single_test = evaluate(single_model, test_loader, DEVICE, multitask=False)
multi_test = evaluate(multi_model, test_loader, DEVICE, multitask=True)

intent_label_names = list(intent_encoder.classes_)

print('\nSingle-task — classification report (intent):')
print(classification_report(
    single_test['_intent_labels'], single_test['_intent_preds'],
    target_names=intent_label_names, zero_division=0,
))

print('\nMulti-task — classification report (intent):')
print(classification_report(
    multi_test['_intent_labels'], multi_test['_intent_preds'],
    target_names=intent_label_names, zero_division=0,
))

comparison_rows = []
for name, m in [('single_task_intent', single_test), ('multi_task_intent_topic', multi_test)]:
    comparison_rows.append({
        'model': name,
        'intent_accuracy': m['intent_accuracy'],
        'intent_f1_macro': m['intent_f1_macro'],
        'intent_f1_micro': m['intent_f1_micro'],
        'intent_f1_weighted': m['intent_f1_weighted'],
        'topic_accuracy': m.get('topic_accuracy', np.nan),
        'topic_f1_macro': m.get('topic_f1_macro', np.nan),
    })
comparison_df = pd.DataFrame(comparison_rows)
print('\nСравнительная таблица (test):')
print(comparison_df.to_string(index=False))

metrics_csv = TABLES_DIR / 'dialogsum_ru_multitask_intent_topic_metrics.csv'
try:
    metrics_csv.parent.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(metrics_csv, index=False)
    print(f'Сохранено: {metrics_csv}')
except OSError as e:
    print(f'Не удалось сохранить {metrics_csv}: {e}')


## Ячейка 10 — Анализ ошибок и визуализации

Строим confusion matrix для intent multi-task модели и сохраняем датафрейм с предсказаниями.

In [ ]:
# cell 10: error analysis and visualizations
intent_label_names = list(intent_encoder.classes_)
cm = confusion_matrix(
    multi_test['_intent_labels'], multi_test['_intent_preds'],
    labels=list(range(NUM_INTENTS)),
)
fig, ax = plt.subplots(figsize=(max(8, NUM_INTENTS * 0.6), max(6, NUM_INTENTS * 0.5)))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=intent_label_names, yticklabels=intent_label_names, ax=ax,
)
ax.set_xlabel('Предсказанный intent')
ax.set_ylabel('Истинный intent')
ax.set_title('Confusion matrix — multi-task intent (test)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

cm_path = FIGURES_DIR / 'multitask_intent_confusion_matrix.png'
try:
    cm_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    print(f'Confusion matrix сохранена: {cm_path}')
except OSError as e:
    print(f'Не удалось сохранить {cm_path}: {e}')
plt.show()

# Таблица с предсказаниями (multi-task) на тестовом сете
pred_df = test_df.copy().reset_index(drop=True)
pred_df['intent_pred_id'] = multi_test['_intent_preds']
pred_df['intent_pred'] = intent_encoder.inverse_transform(pred_df['intent_pred_id'])
if multi_test['_topic_preds']:
    pred_df['topic_pred_id'] = multi_test['_topic_preds']
    pred_df['cluster_id_pred'] = topic_encoder.inverse_transform(pred_df['topic_pred_id'])

pred_csv = TABLES_DIR / 'dialogsum_ru_multitask_intent_topic_predictions.csv'
try:
    pred_csv.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_csv(pred_csv, index=False)
    print(f'Предсказания сохранены: {pred_csv} ({len(pred_df)} строк)')
except OSError as e:
    print(f'Не удалось сохранить {pred_csv}: {e}')

errors = pred_df[pred_df['intent_label'] != pred_df['intent_pred']]
print(f'\nВсего ошибок intent на test: {len(errors)} / {len(pred_df)} ({len(errors)/max(len(pred_df),1):.2%})')
if len(errors):
    print('Примеры ошибок:')
    print(errors[['utterance_text', 'intent_label', 'intent_pred']].head(10).to_string(index=False))


## Ячейка 11 — Сохранение артефактов

Сохраняем state_dict обеих моделей, label encoders и конфигурацию запуска.

In [ ]:
# cell 11: save artifacts
config_to_save = {
    'model_name': MODEL_NAME,
    'max_len': MAX_LEN,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'lambda_intent': LAMBDA_INTENT,
    'lambda_topic': LAMBDA_TOPIC,
    'top_n_clusters': TOP_N_CLUSTERS,
    'min_intent_count': MIN_INTENT_COUNT,
    'max_samples': MAX_SAMPLES,
    'freeze_encoder': FREEZE_ENCODER,
    'random_state': RANDOM_STATE,
    'num_intents': NUM_INTENTS,
    'num_topics': NUM_TOPICS,
    'intent_classes': list(intent_encoder.classes_),
    'topic_classes': [int(x) for x in topic_encoder.classes_],
    'used_fallback_weak_labels': bool(USED_FALLBACK),
    'device': str(DEVICE),
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
}

try:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(single_model.state_dict(), MODELS_DIR / 'single_task_intent_model.pt')
    print(f"Сохранена single-task модель: {MODELS_DIR / 'single_task_intent_model.pt'}")
except OSError as e:
    print(f'Не удалось сохранить single-task модель: {e}')

try:
    torch.save(multi_model.state_dict(), MODELS_DIR / 'multitask_intent_topic_model.pt')
    print(f"Сохранена multi-task модель: {MODELS_DIR / 'multitask_intent_topic_model.pt'}")
except OSError as e:
    print(f'Не удалось сохранить multi-task модель: {e}')

try:
    joblib.dump(intent_encoder, MODELS_DIR / 'intent_label_encoder.joblib')
    joblib.dump(topic_encoder, MODELS_DIR / 'topic_label_encoder.joblib')
    print('Сохранены label encoders.')
except Exception as e:
    print(f'Не удалось сохранить label encoders: {e}')

try:
    with open(MODELS_DIR / 'multitask_config.json', 'w', encoding='utf-8') as f:
        json.dump(config_to_save, f, ensure_ascii=False, indent=2)
    print(f"Сохранён конфиг: {MODELS_DIR / 'multitask_config.json'}")
except OSError as e:
    print(f'Не удалось сохранить конфиг: {e}')


## Ячейка 12 — Выводы и обсуждение

### Что было сделано

1. Загружены utterance-уровневые данные DialogSum-RU с **intent-метками** (`intent_label`)
   и **тематическими кластерами** (`cluster_id`) из эмбеддинговой кластеризации
   блокнота `08`. При отсутствии готового weak-датасета он восстанавливается из
   topic clusters и rule-based intent-разметки.
2. Описана и обучена **multi-task нейросеть** на базе `transformers.AutoModel`
   (по умолчанию — `paraphrase-multilingual-MiniLM-L12-v2`) с общим энкодером,
   общей проекцией и двумя головами: для intent и для topic-cluster.
3. Параллельно обучён **single-task baseline**, использующий тот же энкодер, но без
   topic-головы. Это контролирует, что разница в метриках вызвана именно
   многозадачностью, а не разной архитектурой.
4. Произведено сравнение моделей по `accuracy`, `F1 macro/micro/weighted` для intent
   и `accuracy/F1 macro` для topic. Результаты сохранены в
   `dialogsum_ru_multitask_intent_topic_metrics.csv`.
5. Построена confusion matrix intent для multi-task модели; таблица предсказаний
   сохранена в `dialogsum_ru_multitask_intent_topic_predictions.csv`.

### Интерпретация результатов

- Topic-задача служит **индуктивным смещением**: модель не может игнорировать тематику
  реплики, и это часто помогает intent-голове в условиях слабых меток.
- Сравнивая `intent_f1_macro` single vs multi, можно понять, дала ли вспомогательная
  задача прирост. Если прирост положительный — гипотеза подтверждается, если
  отрицательный — `LAMBDA_TOPIC` слишком велик и подавляет основную задачу.
- Шумовой кластер (`cluster_id == -1`) исключён, а из остальных взяты только top-N
  крупнейших, чтобы topic-голова не вырождалась из-за длиннохвостого распределения.

### Ограничения

- **Слабая разметка intent.** В отсутствие ручной верификации метрики ограничены
  качеством эвристик из `09`/`10` или rule-based fallback'а — модель может выучивать
  закономерности самих эвристик, а не истинных намерений. Содержательные выводы
  требуют ручной валидации (см. `10_intent_manual_validation_dialogsum_ru.ipynb`).
- **Дисбаланс классов.** `other`, `informational_request` и `confirmation` обычно
  доминируют. Macro-F1 поэтому информативнее, чем accuracy.
- **Качество кластеров.** topic-задача настолько хороша, насколько хороша исходная
  кластеризация в `08`. Шумные кластеры ухудшат и многозадачную модель.
- **Малое число эпох.** Для CPU/Colab оставлено `EPOCHS=3`; на GPU имеет смысл
  увеличить и подключить раннюю остановку.

### Возможные расширения

- Заменить энкодер на `DeepPavlov/rubert-base-cased` или `xlm-roberta-base` —
  достаточно поменять `MODEL_NAME`.
- Добавить uncertainty-weighting лоссов (Kendall et al., 2018) вместо ручного выбора
  `LAMBDA_INTENT` / `LAMBDA_TOPIC`.
- Использовать иерархию кластеров (например, агрегированную в `08`) как ещё одну
  вспомогательную голову.
- Подключить активное обучение поверх предсказаний multi-task модели для разметки
  реальных intent-меток.
